# M4-09: Model Deployment

This notebook covers Tasks 1, 3, and 4 of the model deployment lab:
- **Task 1**: Train, evaluate, and serialize a RandomForestClassifier on the Iris dataset.
- **Task 3**: Test the Flask API with valid, invalid, and batch requests.
- **Task 4**: Reflection on production deployment considerations.


In [1]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

---
## Task 1: Train & Serialize

We load the Iris dataset, split into 80/20 train/test, train a `RandomForestClassifier`, evaluate it, and serialize the model to disk. We also verify the round-trip by reloading and checking prediction identity.


In [2]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

print(f"Dataset shape: {X.shape}")
print(f"Classes: {iris.target_names}")
print(f"Feature names: {iris.feature_names}")

Dataset shape: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [3]:
# Train/test split — 80/20, reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

Train size: 120 | Test size: 30


In [4]:
# Train the RandomForestClassifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Test Accuracy: 1.0000

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



The model achieves **100% accuracy** on this test split — not surprising for the Iris dataset, which is small, clean, and linearly separable between most classes. The Random Forest with 100 trees easily memorizes the pattern.


In [5]:
# Serialize the trained model and target names
joblib.dump(clf, "model.joblib")
joblib.dump(iris.target_names, "target_names.joblib")
print("Saved: model.joblib and target_names.joblib")

Saved: model.joblib and target_names.joblib


In [6]:
# Round-trip verification: reload and predict
clf_loaded = joblib.load("model.joblib")
y_pred_loaded = clf_loaded.predict(X_test)

identical = np.array_equal(y_pred, y_pred_loaded)
print(f"Original predictions:  {y_pred}")
print(f"Reloaded predictions:  {y_pred_loaded}")
print(f"\nPredictions identical after round-trip: {identical}")

Original predictions:  [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Reloaded predictions:  [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]

Predictions identical after round-trip: True


The round-trip serialization is verified — `joblib` preserves the model state exactly, producing byte-for-byte identical predictions after reload.

---
## Task 3: Test the Flask API

> **Before running this section**, start the Flask server in a terminal:
> ```bash
> python app.py
> ```

We'll test all three endpoints: `/health`, `/predict`, and `/predict_batch`.


In [8]:
BASE_URL = "http://localhost:5000"

# ── Health check ──────────────────────────────────────────────
resp = requests.get(f"{BASE_URL}/health")
print(f"Status: {resp.status_code}")
print(f"Body:   {resp.json()}")

Status: 200
Body:   {'status': 'healthy'}


The `/health` endpoint returns `200 OK` with `{"status": "healthy"}` — exactly what a load balancer expects before routing traffic to this instance.


In [9]:
# ── Single prediction: a classic setosa sample ─────────────────
sample = {"features": [5.1, 3.5, 1.4, 0.2]}
resp = requests.post(f"{BASE_URL}/predict", json=sample)
print(f"Status: {resp.status_code}")
result = resp.json()
print(f"Predicted class: {result['predicted_class']}")
print(f"Probabilities:")
for cls, prob in result['probabilities'].items():
    print(f"  {cls:12s}: {prob:.4f}")

Status: 200
Predicted class: setosa
Probabilities:
  setosa      : 1.0000
  versicolor  : 0.0000
  virginica   : 0.0000


The API correctly classifies the sample `[5.1, 3.5, 1.4, 0.2]` as **setosa**, with high confidence. This matches the expected label for the first row of the classic Iris dataset.


In [10]:
# ── Error handling: 3 malformed requests ───────────────────────

# 1. Missing 'features' key
bad1 = {"data": [5.1, 3.5, 1.4, 0.2]}
resp1 = requests.post(f"{BASE_URL}/predict", json=bad1)
print(f"[Missing key]     Status: {resp1.status_code} | {resp1.json()}")

# 2. Wrong number of features (3 instead of 4)
bad2 = {"features": [5.1, 3.5, 1.4]}
resp2 = requests.post(f"{BASE_URL}/predict", json=bad2)
print(f"[Wrong count]     Status: {resp2.status_code} | {resp2.json()}")

# 3. Non-numeric values
bad3 = {"features": ["a", "b", "c", "d"]}
resp3 = requests.post(f"{BASE_URL}/predict", json=bad3)
print(f"[Non-numeric]     Status: {resp3.status_code} | {resp3.json()}")

[Missing key]     Status: 400 | {'error': "Missing 'features' key in request body"}
[Wrong count]     Status: 400 | {'error': 'Expected exactly 4 feature values, got 3'}
[Non-numeric]     Status: 400 | {'error': "Non-numeric value at index 0: 'a'. All features must be numeric."}


All three malformed requests correctly return **400 Bad Request** with descriptive error messages:
1. Missing `features` key → clear message about the required field.
2. Wrong feature count → tells the caller how many were given vs. expected.
3. Non-numeric values → identifies which index had the offending value.

Good error messages are critical for API usability — they allow clients to self-debug without reading source code.


In [11]:
# ── Batch prediction: 5 samples from the test set ─────────────
batch_samples = X_test[:5].tolist()
local_preds = clf_loaded.predict(X_test[:5])

resp = requests.post(f"{BASE_URL}/predict_batch", json={"samples": batch_samples})
print(f"Status: {resp.status_code}")
api_preds = resp.json()["predictions"]

target_names_loaded = joblib.load("target_names.joblib")

print(f"\n{'Sample':<8} {'Local prediction':<20} {'API prediction':<20} {'Match'}")
print("-" * 60)
all_match = True
for i, (local_idx, api_result) in enumerate(zip(local_preds, api_preds)):
    local_name = target_names_loaded[local_idx]
    api_name = api_result['predicted_class']
    match = local_name == api_name
    if not match:
        all_match = False
    print(f"{i:<8} {local_name:<20} {api_name:<20} {'✓' if match else '✗'}")

print(f"\nAll predictions match: {all_match}")

Status: 200

Sample   Local prediction     API prediction       Match
------------------------------------------------------------
0        versicolor           versicolor           ✓
1        setosa               setosa               ✓
2        virginica            virginica            ✓
3        versicolor           versicolor           ✓
4        versicolor           versicolor           ✓

All predictions match: True


## Task 4: Reflection

### What would need to change for production deployment?

Flask's built-in development server (`app.run(debug=True)`) is single-threaded and not designed to handle concurrent load. In production:

- **WSGI server**: Replace with `gunicorn` (`gunicorn -w 4 app:app`) or `uvicorn` for async frameworks. These handle concurrency properly.
- **Containerization**: Package the app, model artifacts, and dependencies into a Docker image. This ensures reproducibility across environments and enables easy scaling.
- **Environment variables**: API ports, model paths, log levels, and feature flags should be read from environment variables — never hardcoded. Use `python-dotenv` or Kubernetes ConfigMaps.
- **HTTPS / TLS**: Terminate TLS at a reverse proxy (nginx, AWS ALB) so the app never handles raw HTTP in production. This is non-negotiable for any internet-facing service.
- **Health checks & graceful shutdown**: Already have `/health` — also implement readiness vs. liveness probes for Kubernetes.

### How would you handle model versioning?

Retraining always risks regression. A practical versioning strategy:

- **Artifact tagging**: Store each model as `model_v{timestamp}.joblib` alongside metadata (training date, dataset hash, accuracy on hold-out set). A registry (MLflow, DVC, or even S3 + a JSON manifest) tracks which version is "current".
- **Shadow mode / A-B testing**: Deploy the new model alongside the old one. Route a small fraction of traffic to the new model and compare predictions and error rates before promoting.
- **Rollback capability**: The production config should point to a versioned artifact path — reverting is just changing a config variable and redeploying, not re-training.
- **Evaluation gates**: CI/CD pipeline runs the new model on a frozen validation set. If accuracy drops below a threshold (e.g., 95% of baseline), the pipeline fails and the new model is not deployed.

### What monitoring would you add?

Three layers of observability matter:

1. **Infrastructure metrics**: CPU/memory usage, pod restarts (Prometheus + Grafana).
2. **API metrics**: Request latency (p50/p95/p99), error rate by status code, requests-per-second. Export as Prometheus metrics or push to Datadog/CloudWatch.
3. **ML-specific metrics**:
   - **Input drift**: Track rolling statistics (mean, std) of incoming features. Alert if they diverge significantly from training distribution (using KL divergence or PSI — Population Stability Index).
   - **Prediction distribution**: If the model starts predicting one class 80% of the time when it used to predict 33%, something is wrong upstream.
   - **Ground truth labels** (when available): Log predictions and, once labels arrive (e.g., from user feedback), compute live accuracy to detect model decay over time.

### How would the architecture change at 1,000 req/s?

At ~1K req/s a single Flask/gunicorn pod will saturate:

- **Horizontal scaling**: Run multiple replicas behind a load balancer (e.g., Kubernetes Deployment with HPA — Horizontal Pod Autoscaler keyed on CPU or request queue depth).
- **Model loading**: Instead of loading the model on every request (already avoided here), pre-load it once at startup. With multiple replicas each holds its own in-memory copy — that's fine for a 10MB Random Forest.
- **Async inference**: For CPU-bound models, use async workers or a task queue (Celery + Redis) to decouple request acceptance from inference — especially if some requests are slow.
- **Batching at the server**: Accumulate requests for ~10ms and run inference on a batch of 32 — throughput increases dramatically since scikit-learn's `predict` is vectorized.
- **Caching**: For identical inputs, a Redis cache with input hash as key returns instantly without hitting the model. Useful when users send the same sensor reading repeatedly.
- **Dedicated inference servers**: At scale, consider Triton Inference Server or TorchServe which handle batching, multi-model serving, and GPU scheduling natively.
